In [0]:
# Databricks notebook source
# DBTITLE 1,Imports & Config
import hashlib
import json
import logging
import os

import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)

# ── Paths ─────────────────────────────────────────────────────────────────────
OUTPUTS_DIR        = "/Volumes/movie_recsys/data/outputs"
REVIEWS_PATH       = f"{OUTPUTS_DIR}/reviews_5core.parquet"
COHORT_NOVA_PATH   = f"{OUTPUTS_DIR}/cohort_nova.parquet"

# ── Config params (all overridable) ───────────────────────────────────────────
ENROLLMENT_MONTH   = "202604"         # April 2026 — blueprint spec
MIN_LIFETIME_RATINGS = 25             # blueprint: sample users with 25+ ratings
ONBOARDING_DAYS    = 30               # observation window
GRAD_THRESHOLD     = 10               # CB → CF switch threshold (blueprint)
RANDOM_STATE       = 42
TARGET_COHORT_SIZE = 10_000           # max users in cohort

# A/B params
TREATMENT_RETENTION_LIFT = 0.08       # treatment arm retains 8pp more at day 30
CONTROL_DAY30_RETENTION  = 0.40       # 40% baseline retention (conservative)

In [0]:
log.info("Loading reviews from %s", REVIEWS_PATH)
reviews = pd.read_parquet(
    REVIEWS_PATH,
    columns=["user_id", "parent_asin", "rating", "helpful_vote", "event_ts"]
)

assert "parent_asin" in reviews.columns, "reviews missing parent_asin"
assert "event_ts"    in reviews.columns, "reviews missing event_ts"

# Users with 25+ lifetime ratings (blueprint spec)
lifetime_counts = reviews.groupby("user_id").size()
eligible_users  = lifetime_counts[lifetime_counts >= MIN_LIFETIME_RATINGS].index.tolist()

log.info("Total users in reviews    : %d", reviews["user_id"].nunique())
log.info("Users with 25+ ratings    : %d", len(eligible_users))

# Sample cohort (cap at TARGET_COHORT_SIZE)
rng = np.random.default_rng(RANDOM_STATE)
if len(eligible_users) > TARGET_COHORT_SIZE:
    cohort_users = rng.choice(eligible_users, size=TARGET_COHORT_SIZE, replace=False).tolist()
else:
    cohort_users = eligible_users

log.info("Cohort size               : %d", len(cohort_users))

In [0]:
cohort_reviews = reviews[reviews["user_id"].isin(cohort_users)].copy()
cohort_reviews["event_ts"] = pd.to_datetime(cohort_reviews["event_ts"])

# Anchor day 0 = each user's first ever rating timestamp
first_ts = cohort_reviews.groupby("user_id")["event_ts"].min().rename("anchor_ts")
cohort_reviews = cohort_reviews.merge(first_ts, on="user_id")

cohort_reviews["days_since_first"] = (
    (cohort_reviews["event_ts"] - cohort_reviews["anchor_ts"])
    .dt.total_seconds() / 86400
).round(2)

# Keep only activity within the 30-day onboarding window
onboarding = cohort_reviews[cohort_reviews["days_since_first"] <= ONBOARDING_DAYS].copy()

log.info("Interactions in onboarding window : %d", len(onboarding))
log.info("Users with any onboarding activity: %d", onboarding["user_id"].nunique())

In [0]:
# enrollment_date is fixed as April 2026 for all cohort users (blueprint spec)
ENROLLMENT_DATE = pd.Timestamp("2026-04-01")

# Deterministic A/B assignment via user_id hash (blueprint spec)
def ab_assign(user_id: str) -> str:
    digest = int(hashlib.md5(user_id.encode()).hexdigest(), 16)
    return "treatment" if digest % 2 == 0 else "control"

cohort_df = pd.DataFrame({"user_id": cohort_users})
cohort_df["enrollment_date"] = ENROLLMENT_DATE
cohort_df["ab_group"]        = cohort_df["user_id"].apply(ab_assign)

ab_counts = cohort_df["ab_group"].value_counts()
log.info("A/B split — control: %d  treatment: %d",
         ab_counts.get("control", 0), ab_counts.get("treatment", 0))

In [0]:
# Rating count during onboarding window (drives CB→CF graduation)
rating_counts = (
    onboarding.groupby("user_id")
    .size()
    .reset_index(name="onboarding_rating_count")
)

# Day of last activity (proxy for retention)
last_active = (
    onboarding.groupby("user_id")["days_since_first"]
    .max()
    .reset_index(name="last_active_day")
)

# Genre diversity (number of distinct items interacted with)
item_diversity = (
    onboarding.groupby("user_id")["parent_asin"]
    .nunique()
    .reset_index(name="distinct_items")
)

# Mean rating given during onboarding
mean_rating = (
    onboarding.groupby("user_id")["rating"]
    .mean()
    .reset_index(name="mean_rating_onboarding")
)

# Merge all features onto cohort
cohort_df = (
    cohort_df
    .merge(rating_counts,  on="user_id", how="left")
    .merge(last_active,    on="user_id", how="left")
    .merge(item_diversity, on="user_id", how="left")
    .merge(mean_rating,    on="user_id", how="left")
)

# Fill users with zero activity in window
cohort_df["onboarding_rating_count"] = cohort_df["onboarding_rating_count"].fillna(0).astype(int)
cohort_df["last_active_day"]         = cohort_df["last_active_day"].fillna(0)
cohort_df["distinct_items"]          = cohort_df["distinct_items"].fillna(0).astype(int)
cohort_df["mean_rating_onboarding"]  = cohort_df["mean_rating_onboarding"].fillna(0)

# CB→CF graduation flag
cohort_df["graduated_to_cf"] = cohort_df["onboarding_rating_count"] >= GRAD_THRESHOLD

log.info("Users who graduated to CF (>= %d ratings): %d / %d (%.1f%%)",
         GRAD_THRESHOLD,
         cohort_df["graduated_to_cf"].sum(),
         len(cohort_df),
         100 * cohort_df["graduated_to_cf"].mean())

In [0]:
# Retention logic (blueprint spec):
#   Control  : popularity-based recs → base retention rate
#   Treatment: personalised CB/CF    → base + lift
#
# A user is "retained" if they were active past day 15 in their onboarding window
# (proxy: last_active_day > 15). Then we apply group-level retention probability
# to determine the final retained_day30 flag.

rng2 = np.random.default_rng(RANDOM_STATE + 1)

def simulate_retention(row):
    # Base signal: was the user still active after day 15?
    active_signal = float(row["last_active_day"] > 15)

    # Group retention probability
    if row["ab_group"] == "treatment":
        p_retain = CONTROL_DAY30_RETENTION + TREATMENT_RETENTION_LIFT
    else:
        p_retain = CONTROL_DAY30_RETENTION

    # Blend activity signal with group probability
    # Active users lean toward retention; inactive users lean away
    p_individual = 0.6 * p_retain + 0.4 * active_signal
    p_individual = float(np.clip(p_individual, 0.0, 1.0))
    return p_individual

cohort_df["retention_probability"] = cohort_df.apply(simulate_retention, axis=1)
cohort_df["retained_day30"] = (
    rng2.random(len(cohort_df)) < cohort_df["retention_probability"]
).astype(int)

# Log observed retention by group
for grp in ["control", "treatment"]:
    mask = cohort_df["ab_group"] == grp
    obs  = cohort_df.loc[mask, "retained_day30"].mean()
    log.info("Observed day-30 retention — %s: %.1f%%", grp, 100 * obs)

In [0]:
cohort_df["movie_id"] = cohort_df.merge(
    onboarding.groupby("user_id")["parent_asin"].first().reset_index(),
    on="user_id", how="left"
)["parent_asin"].fillna("unknown")

# Final column order (matches blueprint section 3.4 schema)
cohort_nova = cohort_df[[
    "user_id",
    "enrollment_date",
    "ab_group",
    "onboarding_rating_count",
    "graduated_to_cf",
    "last_active_day",
    "distinct_items",
    "mean_rating_onboarding",
    "retention_probability",
    "retained_day30",
    "movie_id",
]].copy()

os.makedirs(OUTPUTS_DIR, exist_ok=True)
cohort_nova.to_parquet(COHORT_NOVA_PATH, index=False)
log.info("cohort_nova saved → %s  (%d rows)", COHORT_NOVA_PATH, len(cohort_nova))

In [0]:
print("=" * 65)
print("JOB 3 VALIDATION")
print("=" * 65)

results = {}
def check(name, passed, detail=""):
    results[name] = passed
    tag = "✅" if passed else "❌"
    print(f"  {tag}  {name}" + (f"  [{detail}]" if detail else ""))

cn = pd.read_parquet(COHORT_NOVA_PATH)

print("\nT1 · File & size")
check("cohort_nova.parquet exists",  os.path.exists(COHORT_NOVA_PATH))
check("Has >= 5,000 users",          len(cn) >= 5_000,              f"{len(cn):,}")

print("\nT2 · Schema")
required_cols = [
    "user_id", "enrollment_date", "ab_group",
    "onboarding_rating_count", "graduated_to_cf",
    "last_active_day", "retained_day30", "retention_probability",
]
for col in required_cols:
    check(f"Has column: {col}", col in cn.columns)

print("\nT3 · A/B balance")
ab = cn["ab_group"].value_counts()
ratio = ab.min() / ab.max()
check("A/B split roughly 50/50",
      ratio >= 0.45,
      f"control={ab.get('control',0):,}  treatment={ab.get('treatment',0):,}")

print("\nT4 · Retention signal")
ctrl_ret  = cn[cn["ab_group"] == "control"]["retained_day30"].mean()
treat_ret = cn[cn["ab_group"] == "treatment"]["retained_day30"].mean()
lift      = treat_ret - ctrl_ret
check("Control retention > 0",       ctrl_ret > 0,   f"{ctrl_ret:.1%}")
check("Treatment retention > ctrl",  treat_ret > ctrl_ret,
      f"treatment={treat_ret:.1%}  control={ctrl_ret:.1%}  lift={lift:+.1%}")

print("\nT5 · Graduation")
grad_pct = cn["graduated_to_cf"].mean()
check("Some users graduated to CF",  grad_pct > 0,  f"{grad_pct:.1%}")

print("\nT6 · No nulls on key columns")
for col in ["user_id", "ab_group", "retained_day30", "retention_probability"]:
    null_count = cn[col].isna().sum()
    check(f"No nulls in {col}", null_count == 0, f"{null_count} nulls")

print("\n" + "=" * 65)
passed = sum(results.values())
failed = len(results) - passed
print(f"RESULT: {passed} passed, {failed} failed")
if failed == 0:
    print(f"✅ JOB 3 COMPLETE — cohort_nova: {len(cn):,} users")
    print(f"   Control retention:   {ctrl_ret:.1%}")
    print(f"   Treatment retention: {treat_ret:.1%}")
    print(f"   Lift:                {lift:+.1%}")
    print("Proceed to Job 4 (A/B simulation & LTV).")
else:
    print(f"❌ {failed} check(s) failed — see above.")
print("=" * 65)